# Task 2.1: Feature Extraction for KNN Baseline

**Owner:** Ahmed  
**Dependencies:** Task 1.4 (Training pairs created)  
**Estimated Time:** 3 hours

## Objective
Extract features for all places in the dataset to use with KNN baseline model. We'll implement two feature extraction strategies:

### Option A: Simple Features
- One-hot encoded metadata (Time of Day, Country)
- TF-IDF text vectors from descriptions (~200 features)
- **Total dimensions:** ~250-300 features

### Option B: Pre-trained Features
- ResNet50 image embeddings (2048-dim, frozen)
- Sentence-BERT text embeddings (384-dim using 'all-MiniLM-L6-v2')
- **Total dimensions:** 2432 features

## Deliverables
- `data/features/places_features_simple.npy`
- `data/features/places_features_pretrained.npy`
- `data/features/feature_metadata.json`

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
import torch
from torchvision import models, transforms
from sentence_transformers import SentenceTransformer
from PIL import Image
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

C:\Users\qadim\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
PyTorch version: 2.7.1+cu118
CUDA available: True


## 2. Load Data

In [2]:
# Define paths
project_root = Path.cwd().parent
data_dir = project_root / 'data'
processed_dir = data_dir / 'processed'
features_dir = data_dir / 'features'
raw_dir = data_dir / 'raw'

# Create features directory if it doesn't exist
features_dir.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")
print(f"Features directory: {features_dir}")

Project root: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system
Data directory: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data
Features directory: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features


In [3]:
# Load cleaned dataset
cleaned_data = pd.read_csv(processed_dir / 'cleaned_dataset.csv')
print(f"Loaded {len(cleaned_data)} records from cleaned dataset")
print(f"\nColumns: {list(cleaned_data.columns)}")
print(f"\nFirst few rows:")
cleaned_data.head()

Loaded 955 records from cleaned dataset

Columns: ['source_file', 'Image URL', 'Description', 'Country', 'Country_Standardized', 'Time of Day', 'Time_of_Day_Standardized', 'preference']

First few rows:


,source_file,Image URL,Description,Country,Country_Standardized,Time of Day,Time_of_Day_Standardized,preference
0,2936035-1161937.csv,https://commons.wikimedia.org/wiki/File:Dom_of...,a clear image of the dome of the rock in jerus...,palestine,Palestine,afternoon,Afternoon,1
1,2936035-1161937.csv,https://upload.wikimedia.org/wikipedia/commons...,a clear image of the ibrahimi mosque (cave of ...,palestine,Palestine,morning,Morning,1
2,2936035-1161937.csv,https://upload.wikimedia.org/wikipedia/commons...,a clear image of the ancient ruins in sebastia...,palestine,Palestine,afternoon,Afternoon,1
3,2936035-1161937.csv,https://upload.wikimedia.org/wikipedia/commons...,a clear image of mar saba monastery in bethleh...,palestine,Palestine,afternoon,Afternoon,1
4,2936035-1161937.csv,https://upload.wikimedia.org/wikipedia/commons...,a clear aerial view of tell es-sultan in jeric...,palestine,Palestine,morning,Morning,1


In [4]:
# Load image path mapping
image_mapping = pd.read_csv(processed_dir / 'image_path_mapping.csv')
print(f"Loaded {len(image_mapping)} image path mappings")
print(f"\nColumns: {list(image_mapping.columns)}")
image_mapping.head()

Loaded 794 image path mappings

Columns: ['Image URL', 'local_image_path']


,Image URL,local_image_path
0,https://upload.wikimedia.org/wikipedia/commons...,C:\Users\qadim\OneDrive - student.birzeit.edu\...
1,https://upload.wikimedia.org/wikipedia/commons...,C:\Users\qadim\OneDrive - student.birzeit.edu\...
2,https://upload.wikimedia.org/wikipedia/commons...,C:\Users\qadim\OneDrive - student.birzeit.edu\...
3,https://upload.wikimedia.org/wikipedia/commons...,C:\Users\qadim\OneDrive - student.birzeit.edu\...
4,https://upload.wikimedia.org/wikipedia/commons...,C:\Users\qadim\OneDrive - student.birzeit.edu\...


## 3. Data Preparation

In [6]:
# Get unique places (we want one feature vector per place)
# Each unique Image URL represents a unique place
places = cleaned_data.drop_duplicates(subset=['Image URL']).copy()
places = places.reset_index(drop=True)

print(f"Total unique places: {len(places)}")
print(f"\nColumns available:")
for col in places.columns:
    print(f"  - {col}")

Total unique places: 955

Columns available:
  - source_file
  - Image URL
  - Description
  - Country
  - Country_Standardized
  - Time of Day
  - Time_of_Day_Standardized
  - preference


In [7]:
# Check data quality
print("Data quality checks:")
print(f"Missing descriptions: {places['Description'].isna().sum()}")
print(f"Missing Time_of_Day_Standardized: {places['Time_of_Day_Standardized'].isna().sum()}")
print(f"Missing Country_Standardized: {places['Country_Standardized'].isna().sum()}")

print(f"\nTime of Day categories: {sorted(places['Time_of_Day_Standardized'].unique())}")
print(f"Number of countries: {places['Country_Standardized'].nunique()}")
print(f"Top 10 countries:\n{places['Country_Standardized'].value_counts().head(10)}")

Data quality checks:
Missing descriptions: 0
Missing Time_of_Day_Standardized: 0
Missing Country_Standardized: 0

Time of Day categories: ['Afternoon', 'Evening', 'Invalid', 'Morning', 'Night', 'Other']
Number of countries: 89
Top 10 countries:
Country_Standardized
Italy          65
Japan          62
USA            57
France         50
Switzerland    49
Spain          49
Palestine      44
Maldives       35
Greece         35
Turkey         34
Name: count, dtype: int64


## 4. Option A: Simple Features (TF-IDF + One-Hot Metadata)

### 4.1 Extract TF-IDF Features from Text

In [8]:
# Prepare text data - combine Description with other text fields for richer features
text_data = places['Description'].fillna('')

# Add Activity and Mood to enrich text features (optional)
if 'Activity' in places.columns:
    text_data = text_data + ' ' + places['Activity'].fillna('')
if 'Mood/Emotion' in places.columns:
    text_data = text_data + ' ' + places['Mood/Emotion'].fillna('')

print(f"Sample combined text:\n{text_data.iloc[0]}")
print(f"\nTotal text samples: {len(text_data)}")

Sample combined text:
a clear image of the dome of the rock in jerusalem, showcasing its iconic golden dome. it is one of the most important islamic and historical landmarks in palestine.

Total text samples: 955


In [9]:
# Extract TF-IDF features
print("Extracting TF-IDF features...")
tfidf = TfidfVectorizer(
    max_features=200,
    stop_words='english',
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,  # Must appear in at least 2 documents
    max_df=0.95  # Remove words appearing in >95% of documents
)

tfidf_features = tfidf.fit_transform(text_data).toarray()
print(f"✓ TF-IDF features shape: {tfidf_features.shape}")
print(f"  Features: {tfidf_features.shape[1]}")
print(f"  Samples: {tfidf_features.shape[0]}")

# Show top TF-IDF features
feature_names = tfidf.get_feature_names_out()
print(f"\nTop 20 TF-IDF features: {feature_names[:20].tolist()}")

Extracting TF-IDF features...
✓ TF-IDF features shape: (955, 200)
  Features: 200
  Samples: 955

Top 20 TF-IDF features: ['adventure', 'aerial', 'aerial view', 'al', 'alps', 'ancient', 'architecture', 'atmosphere', 'aurora', 'autumn', 'background', 'bali', 'barcelona', 'beach', 'beaches', 'beautiful', 'beauty', 'blue', 'blue sky', 'boats']


### 4.2 Extract One-Hot Encoded Metadata

In [10]:
# Prepare metadata for one-hot encoding
metadata = places[['Time_of_Day_Standardized', 'Country_Standardized']].copy()

# Fill any missing values
metadata['Time_of_Day_Standardized'] = metadata['Time_of_Day_Standardized'].fillna('Unknown')
metadata['Country_Standardized'] = metadata['Country_Standardized'].fillna('Unknown')

print(f"Metadata shape: {metadata.shape}")
print(f"\nTime of Day distribution:\n{metadata['Time_of_Day_Standardized'].value_counts()}")
print(f"\nCountry distribution (top 10):\n{metadata['Country_Standardized'].value_counts().head(10)}")

Metadata shape: (955, 2)

Time of Day distribution:
Time_of_Day_Standardized
Afternoon    442
Evening      260
Morning      225
Night         18
Invalid        7
Other          3
Name: count, dtype: int64

Country distribution (top 10):
Country_Standardized
Italy          65
Japan          62
USA            57
France         50
Switzerland    49
Spain          49
Palestine      44
Maldives       35
Greece         35
Turkey         34
Name: count, dtype: int64


In [12]:
# One-hot encode metadata
print("One-hot encoding metadata...")
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
metadata_features = encoder.fit_transform(metadata)

print(f"✓ One-hot encoded metadata shape: {metadata_features.shape}")
print(f"  Features: {metadata_features.shape[1]}")
print(f"  - Time of Day categories: {len(metadata['Time_of_Day_Standardized'].unique())}")
print(f"  - Country categories: {len(metadata['Country_Standardized'].unique())}")

# Get feature names
metadata_feature_names = encoder.get_feature_names_out()
print(f"\nSample metadata features: {metadata_feature_names[:10].tolist()}")

One-hot encoding metadata...
✓ One-hot encoded metadata shape: (955, 95)
  Features: 95
  - Time of Day categories: 6
  - Country categories: 89

Sample metadata features: ['Time_of_Day_Standardized_Afternoon', 'Time_of_Day_Standardized_Evening', 'Time_of_Day_Standardized_Invalid', 'Time_of_Day_Standardized_Morning', 'Time_of_Day_Standardized_Night', 'Time_of_Day_Standardized_Other', 'Country_Standardized_Afghanistan', 'Country_Standardized_Albania', 'Country_Standardized_Algeria', 'Country_Standardized_Andorra']


### 4.3 Combine Simple Features

In [13]:
# Combine TF-IDF and one-hot metadata
simple_features = np.hstack([tfidf_features, metadata_features])

print(f"✓ Combined simple features shape: {simple_features.shape}")
print(f"  Total features: {simple_features.shape[1]}")
print(f"    - TF-IDF: {tfidf_features.shape[1]}")
print(f"    - Metadata: {metadata_features.shape[1]}")
print(f"  Samples: {simple_features.shape[0]}")

# Basic statistics
print(f"\nFeature statistics:")
print(f"  Mean: {simple_features.mean():.4f}")
print(f"  Std: {simple_features.std():.4f}")
print(f"  Min: {simple_features.min():.4f}")
print(f"  Max: {simple_features.max():.4f}")
print(f"  Sparsity: {(simple_features == 0).sum() / simple_features.size * 100:.2f}%")

✓ Combined simple features shape: (955, 295)
  Total features: 295
    - TF-IDF: 200
    - Metadata: 95
  Samples: 955

Feature statistics:
  Mean: 0.0159
  Std: 0.0994
  Min: 0.0000
  Max: 1.0000
  Sparsity: 96.42%


## 5. Option B: Pre-trained Features (ResNet50 + Sentence-BERT)

### 5.1 Extract Text Features using Sentence-BERT

In [14]:
# Load Sentence-BERT model
print("Loading Sentence-BERT model...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Model loaded: {sentence_model}")
print(f"  Embedding dimension: {sentence_model.get_sentence_embedding_dimension()}")

Loading Sentence-BERT model...
✓ Model loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
  Embedding dimension: 384


In [15]:
# Extract text embeddings
print("\nExtracting Sentence-BERT embeddings...")
descriptions = places['Description'].fillna('').tolist()

text_embeddings = sentence_model.encode(
    descriptions,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"\n✓ Text embeddings shape: {text_embeddings.shape}")
print(f"  Embedding dimension: {text_embeddings.shape[1]}")
print(f"  Samples: {text_embeddings.shape[0]}")

# Statistics
print(f"\nText embedding statistics:")
print(f"  Mean: {text_embeddings.mean():.4f}")
print(f"  Std: {text_embeddings.std():.4f}")
print(f"  Min: {text_embeddings.min():.4f}")
print(f"  Max: {text_embeddings.max():.4f}")


Extracting Sentence-BERT embeddings...


Batches: 100%|██████████| 30/30 [00:00<00:00, 59.48it/s]



✓ Text embeddings shape: (955, 384)
  Embedding dimension: 384
  Samples: 955

Text embedding statistics:
  Mean: -0.0003
  Std: 0.0510
  Min: -0.2334
  Max: 0.2250


### 5.2 Extract Image Features using ResNet50

In [16]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load ResNet50 pre-trained model
print("\nLoading ResNet50 model...")
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
# Remove the final classification layer to get 2048-dim features
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device)
resnet.eval()
print("✓ ResNet50 loaded (frozen, final layer removed)")

# Define image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print("✓ Image preprocessing pipeline ready")

Using device: cuda

Loading ResNet50 model...
✓ ResNet50 loaded (frozen, final layer removed)
✓ Image preprocessing pipeline ready


In [22]:
# Merge places with image paths  
# Remember: image_mapping has 'Image URL' and 'local_image_path' columns
places_with_images = places.merge(
    image_mapping[['Image URL', 'local_image_path']],
    on='Image URL',
    how='left'
)

# Drop duplicates to ensure we have same number of rows as original places
# (in case image_mapping has duplicate Image URLs)
places_with_images = places_with_images.drop_duplicates(subset=['Image URL']).reset_index(drop=True)

print(f"Places with images: {len(places_with_images)}")
print(f"Places with valid image paths: {places_with_images['local_image_path'].notna().sum()}")
print(f"Places without images: {places_with_images['local_image_path'].isna().sum()}")

if places_with_images['local_image_path'].isna().any():
    print("\n⚠ Warning: Some places don't have associated images")
    print("  These will be assigned zero vectors for image embeddings")

Places with images: 955
Places with valid image paths: 696
Places without images: 259

⚠ Warning: Some places don't have associated images
  These will be assigned zero vectors for image embeddings


In [23]:
# Extract image features
print("\nExtracting ResNet50 image features...")
image_embeddings = []
failed_images = 0

with torch.no_grad():
    for idx, row in tqdm(places_with_images.iterrows(), total=len(places_with_images), desc="Processing images"):
        image_path = row['local_image_path']        
        if pd.isna(image_path) or not Path(raw_dir / image_path).exists():
            # Use zero vector for missing images
            image_embeddings.append(np.zeros(2048))
            failed_images += 1
            continue
        
        try:
            # Load and preprocess image
            img = Image.open(image_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            # Extract features
            features = resnet(img_tensor)
            features = features.squeeze().cpu().numpy()
            
            image_embeddings.append(features)
        except Exception as e:
            # Use zero vector for failed images
            image_embeddings.append(np.zeros(2048))
            failed_images += 1

image_embeddings = np.array(image_embeddings)

print(f"\n✓ Image embeddings shape: {image_embeddings.shape}")
print(f"  Embedding dimension: {image_embeddings.shape[1]}")
print(f"  Samples: {image_embeddings.shape[0]}")
print(f"  Failed/missing images: {failed_images}")

# Statistics
print(f"\nImage embedding statistics:")
print(f"  Mean: {image_embeddings.mean():.4f}")
print(f"  Std: {image_embeddings.std():.4f}")
print(f"  Min: {image_embeddings.min():.4f}")
print(f"  Max: {image_embeddings.max():.4f}")


Extracting ResNet50 image features...


Processing images: 100%|██████████| 955/955 [01:08<00:00, 13.94it/s]



✓ Image embeddings shape: (955, 2048)
  Embedding dimension: 2048
  Samples: 955
  Failed/missing images: 261

Image embedding statistics:
  Mean: 0.3247
  Std: 0.4343
  Min: 0.0000
  Max: 7.9102


### 5.3 Combine Pre-trained Features

In [24]:
# Concatenate image and text embeddings
pretrained_features = np.hstack([image_embeddings, text_embeddings])

print(f"✓ Combined pre-trained features shape: {pretrained_features.shape}")
print(f"  Total features: {pretrained_features.shape[1]}")
print(f"    - Image (ResNet50): {image_embeddings.shape[1]}")
print(f"    - Text (Sentence-BERT): {text_embeddings.shape[1]}")
print(f"  Samples: {pretrained_features.shape[0]}")

# Statistics
print(f"\nCombined feature statistics:")
print(f"  Mean: {pretrained_features.mean():.4f}")
print(f"  Std: {pretrained_features.std():.4f}")
print(f"  Min: {pretrained_features.min():.4f}")
print(f"  Max: {pretrained_features.max():.4f}")

✓ Combined pre-trained features shape: (955, 2432)
  Total features: 2432
    - Image (ResNet50): 2048
    - Text (Sentence-BERT): 384
  Samples: 955

Combined feature statistics:
  Mean: 0.2734
  Std: 0.4163
  Min: -0.2334
  Max: 7.9102


## 6. Save Features

In [25]:
# Save simple features
simple_path = features_dir / 'places_features_simple.npy'
np.save(simple_path, simple_features)
print(f"✓ Saved simple features to: {simple_path}")
print(f"  Shape: {simple_features.shape}")
print(f"  Size: {simple_path.stat().st_size / 1024:.2f} KB")

# Save pre-trained features
pretrained_path = features_dir / 'places_features_pretrained.npy'
np.save(pretrained_path, pretrained_features)
print(f"\n✓ Saved pre-trained features to: {pretrained_path}")
print(f"  Shape: {pretrained_features.shape}")
print(f"  Size: {pretrained_path.stat().st_size / 1024:.2f} KB")

✓ Saved simple features to: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\places_features_simple.npy
  Shape: (955, 295)
  Size: 2201.10 KB

✓ Saved pre-trained features to: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\places_features_pretrained.npy
  Shape: (955, 2432)
  Size: 18145.12 KB


In [27]:
# Save place identifiers (to map features back to places)
# Use Image URL as the unique place identifier
place_ids = places[['source_file', 'Image URL']].copy()
place_ids_path = features_dir / 'place_identifiers.csv'
place_ids.to_csv(place_ids_path, index=False)
print(f"✓ Saved place identifiers to: {place_ids_path}")
print(f"  Records: {len(place_ids)}")

✓ Saved place identifiers to: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\place_identifiers.csv
  Records: 955


In [28]:
# Create and save metadata
metadata_info = {
    'creation_date': pd.Timestamp.now().isoformat(),
    'num_places': len(places),
    'simple_features': {
        'total_dimensions': int(simple_features.shape[1]),
        'tfidf_dimensions': int(tfidf_features.shape[1]),
        'metadata_dimensions': int(metadata_features.shape[1]),
        'tfidf_config': {
            'max_features': 200,
            'ngram_range': [1, 2],
            'min_df': 2,
            'max_df': 0.95
        },
        'metadata_categories': {
            'time_of_day': sorted(metadata['Time_of_Day_Standardized'].unique().tolist()),
            'countries': sorted(metadata['Country_Standardized'].unique().tolist())
        },
        'statistics': {
            'mean': float(simple_features.mean()),
            'std': float(simple_features.std()),
            'min': float(simple_features.min()),
            'max': float(simple_features.max()),
            'sparsity': float((simple_features == 0).sum() / simple_features.size)
        }
    },
    'pretrained_features': {
        'total_dimensions': int(pretrained_features.shape[1]),
        'image_dimensions': int(image_embeddings.shape[1]),
        'text_dimensions': int(text_embeddings.shape[1]),
        'image_model': 'ResNet50 (ImageNet pre-trained)',
        'text_model': 'all-MiniLM-L6-v2 (Sentence-BERT)',
        'failed_images': int(failed_images),
        'statistics': {
            'mean': float(pretrained_features.mean()),
            'std': float(pretrained_features.std()),
            'min': float(pretrained_features.min()),
            'max': float(pretrained_features.max())
        }
    }
}

metadata_path = features_dir / 'feature_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata_info, f, indent=2)

print(f"\n✓ Saved feature metadata to: {metadata_path}")
print("\nMetadata summary:")
print(json.dumps(metadata_info, indent=2))


✓ Saved feature metadata to: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\feature_metadata.json

Metadata summary:
{
  "creation_date": "2026-01-11T21:31:07.214018",
  "num_places": 955,
  "simple_features": {
    "total_dimensions": 295,
    "tfidf_dimensions": 200,
    "metadata_dimensions": 95,
    "tfidf_config": {
      "max_features": 200,
      "ngram_range": [
        1,
        2
      ],
      "min_df": 2,
      "max_df": 0.95
    },
    "metadata_categories": {
      "time_of_day": [
        "Afternoon",
        "Evening",
        "Invalid",
        "Morning",
        "Night",
        "Other"
      ],
      "countries": [
        "Afghanistan",
        "Albania",
        "Algeria",
        "Andorra",
        "Antarctica",
        "Arctic",
        "Argentina",
        "Australia",
        "Austria",
        "Bahamas",
        "Bhutan",
        "Bolivia",
        "Bosnia and Herze

## 7. Verification and Quality Checks

In [29]:
# Load saved features to verify
print("Verifying saved features...\n")

loaded_simple = np.load(features_dir / 'places_features_simple.npy')
loaded_pretrained = np.load(features_dir / 'places_features_pretrained.npy')

# Check shapes match
assert loaded_simple.shape == simple_features.shape, "Simple features shape mismatch!"
assert loaded_pretrained.shape == pretrained_features.shape, "Pre-trained features shape mismatch!"

# Check values match
assert np.allclose(loaded_simple, simple_features), "Simple features values mismatch!"
assert np.allclose(loaded_pretrained, pretrained_features), "Pre-trained features values mismatch!"

print("✓ All verification checks passed!")
print(f"\nSimple features: {loaded_simple.shape}")
print(f"Pre-trained features: {loaded_pretrained.shape}")

Verifying saved features...

✓ All verification checks passed!

Simple features: (955, 295)
Pre-trained features: (955, 2432)


In [30]:
# Feature quality checks
print("\nFeature Quality Checks:\n")

# 1. Check for NaN or Inf values
print("1. NaN/Inf checks:")
print(f"   Simple features - NaN: {np.isnan(simple_features).sum()}, Inf: {np.isinf(simple_features).sum()}")
print(f"   Pre-trained features - NaN: {np.isnan(pretrained_features).sum()}, Inf: {np.isinf(pretrained_features).sum()}")

# 2. Check feature variance (features with zero variance are not useful)
simple_var = simple_features.var(axis=0)
pretrained_var = pretrained_features.var(axis=0)
print(f"\n2. Zero-variance features:")
print(f"   Simple features: {(simple_var == 0).sum()} / {len(simple_var)}")
print(f"   Pre-trained features: {(pretrained_var == 0).sum()} / {len(pretrained_var)}")

# 3. Check feature coverage
print(f"\n3. Feature coverage:")
print(f"   Places with features: {len(simple_features)}")
print(f"   Total places in dataset: {len(places)}")
print(f"   Coverage: {len(simple_features) / len(places) * 100:.2f}%")

# 4. Sample similarity check
from sklearn.metrics.pairwise import cosine_similarity
sample_size = min(100, len(simple_features))
sample_indices = np.random.choice(len(simple_features), sample_size, replace=False)

simple_sim = cosine_similarity(simple_features[sample_indices])
pretrained_sim = cosine_similarity(pretrained_features[sample_indices])

print(f"\n4. Similarity analysis (sample of {sample_size} places):")
print(f"   Simple features - Mean cosine similarity: {simple_sim.mean():.4f} (std: {simple_sim.std():.4f})")
print(f"   Pre-trained features - Mean cosine similarity: {pretrained_sim.mean():.4f} (std: {pretrained_sim.std():.4f})")

print("\n✓ Quality checks complete!")


Feature Quality Checks:

1. NaN/Inf checks:
   Simple features - NaN: 0, Inf: 0
   Pre-trained features - NaN: 0, Inf: 0

2. Zero-variance features:
   Simple features: 0 / 295
   Pre-trained features: 0 / 2432

3. Feature coverage:
   Places with features: 955
   Total places in dataset: 955
   Coverage: 100.00%

4. Similarity analysis (sample of 100 places):
   Simple features - Mean cosine similarity: 0.1538 (std: 0.1950)
   Pre-trained features - Mean cosine similarity: 0.3693 (std: 0.3137)

✓ Quality checks complete!


## 8. Summary

In [31]:
print("="*80)
print("TASK 2.1 COMPLETE: Feature Extraction for KNN Baseline")
print("="*80)

print("\n📊 DELIVERABLES:")
print(f"\n1. Simple Features (TF-IDF + Metadata):")
print(f"   File: {features_dir / 'places_features_simple.npy'}")
print(f"   Shape: {simple_features.shape}")
print(f"   Components:")
print(f"     - TF-IDF text features: {tfidf_features.shape[1]} dims")
print(f"     - One-hot metadata: {metadata_features.shape[1]} dims")
print(f"   Total dimensions: {simple_features.shape[1]}")

print(f"\n2. Pre-trained Features (ResNet50 + Sentence-BERT):")
print(f"   File: {features_dir / 'places_features_pretrained.npy'}")
print(f"   Shape: {pretrained_features.shape}")
print(f"   Components:")
print(f"     - ResNet50 image embedding: {image_embeddings.shape[1]} dims")
print(f"     - Sentence-BERT text embedding: {text_embeddings.shape[1]} dims")
print(f"   Total dimensions: {pretrained_features.shape[1]}")

print(f"\n3. Supporting Files:")
print(f"   - Place identifiers: {features_dir / 'place_identifiers.csv'}")
print(f"   - Feature metadata: {features_dir / 'feature_metadata.json'}")

print(f"\n📈 STATISTICS:")
print(f"   Total places extracted: {len(places)}")
print(f"   Failed/missing images: {failed_images}")

print(f"\n✅ NEXT STEPS:")
print(f"   → Proceed to Task 2.2: KNN model training and evaluation")
print(f"   → Use these feature files as input for KNN classifier")
print(f"   → Compare performance between simple and pre-trained features")

print("\n" + "="*80)

TASK 2.1 COMPLETE: Feature Extraction for KNN Baseline

📊 DELIVERABLES:

1. Simple Features (TF-IDF + Metadata):
   File: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\places_features_simple.npy
   Shape: (955, 295)
   Components:
     - TF-IDF text features: 200 dims
     - One-hot metadata: 95 dims
   Total dimensions: 295

2. Pre-trained Features (ResNet50 + Sentence-BERT):
   File: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\places_features_pretrained.npy
   Shape: (955, 2432)
   Components:
     - ResNet50 image embedding: 2048 dims
     - Sentence-BERT text embedding: 384 dims
   Total dimensions: 2432

3. Supporting Files:
   - Place identifiers: c:\Users\qadim\OneDrive - student.birzeit.edu\birzeit\5_year_1\machine_learning\assingment_3_project\travel-recommender-system\data\features\place